In [ ]:
from rdkit import Chem
from rdkit.Chem import AllChem, ResonanceMolSupplier

In [ ]:
methylene_blue = "InChI=1S/C16H18N3S.ClH/c1-18(2)11-5-7-13-15(9-11)20-16-10-12(19(3)4)6-8-14(16)17-13;/h5-10H,1-4H3;1H/q+1;/p-1"

In [ ]:
from rdkit import RDLogger

RDLogger.DisableLog("rdApp.*")
# Parse the molecule from InChI
mol = Chem.MolFromInchi(methylene_blue)
Chem.SanitizeMol(mol)  # Ensure the original is sanitized

# Generate resonance structures
supplier = ResonanceMolSupplier(mol)
res_structs = list(supplier)

print(f"Number of resonance structures: {len(res_structs)}")

# Sanitize each resonance structure
for res in res_structs:
    Chem.SanitizeMol(res)


# Function to compute atom-level Morgan fingerprints
def get_atom_fps(mol, radius=2, nBits=2048):
    fps = []
    for atom_idx in range(mol.GetNumAtoms()):
        fp = AllChem.GetMorganFingerprintAsBitVect(
            mol, radius=radius, fromAtoms=[atom_idx], nBits=nBits
        )
        fps.append(fp)
    return fps


# Compute fingerprints for each resonance structure
all_fps = []
for i, res in enumerate(res_structs):
    fps = get_atom_fps(res)
    all_fps.append(fps)
    print(f"Resonance structure {i + 1}: {len(fps)} atoms")

# Find atoms that have different fingerprints across resonance structures
differing_atoms = []
if len(all_fps) > 1:
    for atom_idx in range(len(all_fps[0])):
        fp0 = all_fps[0][atom_idx]
        differs = False
        for fps in all_fps[1:]:
            if fps[atom_idx] != fp0:
                differs = True
                break
        if differs:
            differing_atoms.append(atom_idx)

if differing_atoms:
    print(f"Atoms with different Morgan fingerprints: {differing_atoms}")
    # Optionally, print atom symbols
    for idx in differing_atoms:
        atom = res_structs[0].GetAtomWithIdx(idx)
        print(f"Atom {idx}: {atom.GetSymbol()}")
else:
    print(
        "All atoms have the same Morgan fingerprints across all resonance structures."
    )